Prompt 1     
Agisci come uno sviluppatore python. Scrivimi un'app python che gestisca la modifica di una foto. Deve avere un'interfaccia utente intuitiva e moderna utilizzando la libreria tkinter. L'utente deve poter caricare l'immagine che sarà visualizzata a schermo. Sotto di essa ci saranno i vari pulsanti per le modifiche, che l'utente potrà controllare in tempo reale:      
- filtro vintage: applica un effetto seppia all'immagine       
- filtro b/n: applica il classico effetto bianco e nero       
- ritaglio dell'immagine      
- luminosità (con slider)      
- contrasto (con slider)       
- temperatura (con slider)      
- specchio       
- rotazione       

Vorrei infine che ci sia la possibilità di salvare l'immagine sia in formato jpg che in png.

In [2]:
"""
Editor di Foto Professionale
Applicazione moderna per la modifica di immagini con interfaccia intuitiva
"""

import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk, ImageEnhance, ImageFilter, ImageOps
import numpy as np
from typing import Optional
import os


class PhotoEditor:
    """Editor di foto con interfaccia grafica moderna"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("Editor di Foto Professionale")
        self.root.geometry("1200x800")
        self.root.configure(bg="#1a1a1a")
        
        # Variabili per le immagini
        self.original_image: Optional[Image.Image] = None
        self.current_image: Optional[Image.Image] = None
        self.display_image: Optional[ImageTk.PhotoImage] = None
        self.crop_mode = False
        self.crop_start = None
        self.crop_rect = None
        
        # Variabili per i filtri
        self.brightness_var = tk.DoubleVar(value=1.0)
        self.contrast_var = tk.DoubleVar(value=1.0)
        self.temperature_var = tk.DoubleVar(value=0)
        
        # Configurazione dello stile moderno
        self.setup_styles()
        
        # Creazione dell'interfaccia
        self.create_widgets()
        
        # Binding degli eventi
        self.brightness_var.trace_add('write', self.apply_adjustments)
        self.contrast_var.trace_add('write', self.apply_adjustments)
        self.temperature_var.trace_add('write', self.apply_adjustments)
        
    def setup_styles(self):
        """Configura gli stili moderni per l'interfaccia"""
        style = ttk.Style()
        style.theme_use('clam')
        
        # Colori moderni
        bg_dark = "#1a1a1a"
        bg_medium = "#2d2d2d"
        bg_light = "#3d3d3d"
        accent = "#4a9eff"
        text_color = "#ffffff"
        
        # Stile per i pulsanti
        style.configure('Modern.TButton',
                       background=bg_medium,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10))
        style.map('Modern.TButton',
                 background=[('active', bg_light), ('pressed', accent)])
        
        # Stile per le etichette
        style.configure('Modern.TLabel',
                       background=bg_dark,
                       foreground=text_color,
                       font=('Segoe UI', 10))
        
        # Stile per i frame
        style.configure('Modern.TFrame',
                       background=bg_dark)
        
        # Stile per gli slider
        style.configure('Modern.Horizontal.TScale',
                       background=bg_dark,
                       troughcolor=bg_medium,
                       borderwidth=0,
                       sliderthickness=20)
        
    def create_widgets(self):
        """Crea tutti i widget dell'interfaccia"""
        # Frame principale
        main_frame = ttk.Frame(self.root, style='Modern.TFrame')
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Area superiore: canvas per l'immagine
        self.canvas_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        self.canvas_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 10))
        
        self.canvas = tk.Canvas(self.canvas_frame,
                               bg="#2d2d2d",
                               highlightthickness=0,
                               cursor="cross")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Testo placeholder
        self.canvas.create_text(
            400, 300,
            text="Carica un'immagine per iniziare",
            fill="#666666",
            font=('Segoe UI', 16),
            tags="placeholder"
        )
        
        # Binding per il crop
        self.canvas.bind("<Button-1>", self.on_crop_start)
        self.canvas.bind("<B1-Motion>", self.on_crop_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_crop_end)
        
        # Area inferiore: controlli
        controls_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        controls_frame.pack(fill=tk.X)
        
        # Frame per i pulsanti principali
        buttons_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        buttons_frame.pack(fill=tk.X, pady=(0, 10))
        
        # Pulsante carica
        load_btn = ttk.Button(buttons_frame,
                             text="📁 Carica Immagine",
                             command=self.load_image,
                             style='Modern.TButton')
        load_btn.pack(side=tk.LEFT, padx=5)
        
        # Pulsante salva JPG
        save_jpg_btn = ttk.Button(buttons_frame,
                                 text="💾 Salva JPG",
                                 command=lambda: self.save_image('jpg'),
                                 style='Modern.TButton')
        save_jpg_btn.pack(side=tk.LEFT, padx=5)
        
        # Pulsante salva PNG
        save_png_btn = ttk.Button(buttons_frame,
                                 text="💾 Salva PNG",
                                 command=lambda: self.save_image('png'),
                                 style='Modern.TButton')
        save_png_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore
        ttk.Separator(buttons_frame, orient=tk.VERTICAL).pack(side=tk.LEFT, fill=tk.Y, padx=10)
        
        # Pulsanti filtri
        vintage_btn = ttk.Button(buttons_frame,
                                text="🎨 Vintage",
                                command=self.apply_vintage,
                                style='Modern.TButton')
        vintage_btn.pack(side=tk.LEFT, padx=5)
        
        bw_btn = ttk.Button(buttons_frame,
                           text="⚫ Bianco e Nero",
                           command=self.apply_bw,
                           style='Modern.TButton')
        bw_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore
        ttk.Separator(buttons_frame, orient=tk.VERTICAL).pack(side=tk.LEFT, fill=tk.Y, padx=10)
        
        # Pulsanti trasformazione
        crop_btn = ttk.Button(buttons_frame,
                             text="✂️ Ritaglia",
                             command=self.toggle_crop_mode,
                             style='Modern.TButton')
        crop_btn.pack(side=tk.LEFT, padx=5)
        
        mirror_btn = ttk.Button(buttons_frame,
                               text="🔄 Specchio",
                               command=self.apply_mirror,
                               style='Modern.TButton')
        mirror_btn.pack(side=tk.LEFT, padx=5)
        
        rotate_btn = ttk.Button(buttons_frame,
                               text="🔃 Ruota 90°",
                               command=self.apply_rotation,
                               style='Modern.TButton')
        rotate_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore
        ttk.Separator(buttons_frame, orient=tk.VERTICAL).pack(side=tk.LEFT, fill=tk.Y, padx=10)
        
        # Pulsante reset
        reset_btn = ttk.Button(buttons_frame,
                              text="↺ Reset",
                              command=self.reset_image,
                              style='Modern.TButton')
        reset_btn.pack(side=tk.LEFT, padx=5)
        
        # Frame per gli slider
        sliders_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        sliders_frame.pack(fill=tk.X)
        
        # Slider luminosità
        self.create_slider(sliders_frame, "☀️ Luminosità", self.brightness_var, 0.0, 2.0, 0)
        
        # Slider contrasto
        self.create_slider(sliders_frame, "◐ Contrasto", self.contrast_var, 0.0, 2.0, 1)
        
        # Slider temperatura
        self.create_slider(sliders_frame, "🌡️ Temperatura", self.temperature_var, -100, 100, 2)
        
    def create_slider(self, parent, label_text, variable, from_, to, column):
        """Crea uno slider con etichetta"""
        frame = ttk.Frame(parent, style='Modern.TFrame')
        frame.grid(row=0, column=column, padx=10, pady=5, sticky='ew')
        parent.columnconfigure(column, weight=1)
        
        label = ttk.Label(frame, text=label_text, style='Modern.TLabel')
        label.pack(anchor='w')
        
        slider = ttk.Scale(frame,
                          from_=from_,
                          to=to,
                          variable=variable,
                          orient=tk.HORIZONTAL,
                          style='Modern.Horizontal.TScale')
        slider.pack(fill=tk.X, pady=5)
        
        # Etichetta valore
        value_label = ttk.Label(frame, text=f"{variable.get():.2f}", style='Modern.TLabel')
        value_label.pack(anchor='e')
        
        def update_label(*args):
            value_label.config(text=f"{variable.get():.2f}")
        
        variable.trace_add('write', update_label)
        
    def load_image(self):
        """Carica un'immagine dal file system"""
        file_path = filedialog.askopenfilename(
            title="Seleziona un'immagine",
            filetypes=[
                ("Immagini", "*.jpg *.jpeg *.png *.bmp *.gif"),
                ("JPEG", "*.jpg *.jpeg"),
                ("PNG", "*.png"),
                ("Tutti i file", "*.*")
            ]
        )
        
        if file_path:
            try:
                self.original_image = Image.open(file_path)
                self.current_image = self.original_image.copy()
                
                # Reset dei controlli
                self.brightness_var.set(1.0)
                self.contrast_var.set(1.0)
                self.temperature_var.set(0)
                
                self.display_current_image()
                self.canvas.delete("placeholder")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile caricare l'immagine:\n{str(e)}")
                
    def display_current_image(self):
        """Visualizza l'immagine corrente sul canvas"""
        if self.current_image is None:
            return
            
        # Calcola le dimensioni per il fit
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        if canvas_width <= 1 or canvas_height <= 1:
            canvas_width = 800
            canvas_height = 600
            
        # Calcola il ridimensionamento mantenendo l'aspect ratio
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height)
        new_width = int(img_width * ratio * 0.9)  # 90% per margini
        new_height = int(img_height * ratio * 0.9)
        
        # Ridimensiona l'immagine
        display_img = self.current_image.resize((new_width, new_height), Image.Resampling.LANCZOS)
        
        # Converti in PhotoImage
        self.display_image = ImageTk.PhotoImage(display_img)
        
        # Pulisci il canvas e mostra l'immagine
        self.canvas.delete("all")
        self.canvas.create_image(
            canvas_width // 2,
            canvas_height // 2,
            image=self.display_image,
            anchor=tk.CENTER,
            tags="image"
        )
        
    def apply_adjustments(self, *args):
        """Applica le regolazioni di luminosità, contrasto e temperatura"""
        if self.original_image is None:
            return
            
        # Parte dall'immagine originale
        img = self.original_image.copy()
        
        # Applica luminosità
        if self.brightness_var.get() != 1.0:
            enhancer = ImageEnhance.Brightness(img)
            img = enhancer.enhance(self.brightness_var.get())
            
        # Applica contrasto
        if self.contrast_var.get() != 1.0:
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(self.contrast_var.get())
            
        # Applica temperatura (modifica del bilanciamento del colore)
        if self.temperature_var.get() != 0:
            img = self.adjust_temperature(img, self.temperature_var.get())
            
        self.current_image = img
        self.display_current_image()
        
    def adjust_temperature(self, image, value):
        """Regola la temperatura del colore dell'immagine"""
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        # Converti in array numpy
        img_array = np.array(image, dtype=np.float32)
        
        # Applica shift di temperatura
        if value > 0:  # Più caldo (più rosso/giallo)
            img_array[:, :, 0] += value * 0.5  # Rosso
            img_array[:, :, 1] += value * 0.3  # Verde
        else:  # Più freddo (più blu)
            img_array[:, :, 2] += abs(value) * 0.5  # Blu
            
        # Clamp values
        img_array = np.clip(img_array, 0, 255)
        
        return Image.fromarray(img_array.astype(np.uint8))
        
    def apply_vintage(self):
        """Applica un filtro vintage (seppia)"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        img = self.current_image.convert('RGB')
        img_array = np.array(img, dtype=np.float32)
        
        # Matrice seppia
        sepia_filter = np.array([
            [0.393, 0.769, 0.189],
            [0.349, 0.686, 0.168],
            [0.272, 0.534, 0.131]
        ])
        
        # Applica il filtro
        sepia_img = img_array @ sepia_filter.T
        sepia_img = np.clip(sepia_img, 0, 255)
        
        self.original_image = Image.fromarray(sepia_img.astype(np.uint8))
        self.apply_adjustments()
        
    def apply_bw(self):
        """Applica il filtro bianco e nero"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.original_image = self.current_image.convert('L').convert('RGB')
        self.apply_adjustments()
        
    def toggle_crop_mode(self):
        """Attiva/disattiva la modalità ritaglio"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.crop_mode = not self.crop_mode
        if self.crop_mode:
            self.canvas.config(cursor="cross")
            messagebox.showinfo("Modalità Ritaglio",
                              "Trascina il mouse sull'immagine per selezionare l'area da ritagliare.\n"
                              "Clicca di nuovo su 'Ritaglia' per applicare.")
        else:
            self.canvas.config(cursor="arrow")
            if self.crop_rect:
                self.apply_crop()
                
    def on_crop_start(self, event):
        """Inizia la selezione per il ritaglio"""
        if not self.crop_mode:
            return
        self.crop_start = (event.x, event.y)
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
    def on_crop_drag(self, event):
        """Disegna il rettangolo di selezione"""
        if not self.crop_mode or not self.crop_start:
            return
            
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
        self.crop_rect = self.canvas.create_rectangle(
            self.crop_start[0], self.crop_start[1],
            event.x, event.y,
            outline='#4a9eff',
            width=2,
            dash=(5, 5)
        )
        
    def on_crop_end(self, event):
        """Finalizza la selezione"""
        if not self.crop_mode or not self.crop_start:
            return
        self.crop_end = (event.x, event.y)
        
    def apply_crop(self):
        """Applica il ritaglio all'immagine"""
        if not self.crop_start or not self.crop_end:
            return
            
        # Calcola le coordinate relative all'immagine
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height) * 0.9
        
        display_width = int(img_width * ratio)
        display_height = int(img_height * ratio)
        
        offset_x = (canvas_width - display_width) // 2
        offset_y = (canvas_height - display_height) // 2
        
        # Converti coordinate canvas in coordinate immagine
        x1 = int((min(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y1 = int((min(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        x2 = int((max(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y2 = int((max(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        
        # Clamp ai limiti dell'immagine
        x1 = max(0, min(x1, img_width))
        y1 = max(0, min(y1, img_height))
        x2 = max(0, min(x2, img_width))
        y2 = max(0, min(y2, img_height))
        
        if x2 > x1 and y2 > y1:
            self.original_image = self.current_image.crop((x1, y1, x2, y2))
            self.apply_adjustments()
            
        self.crop_mode = False
        self.crop_start = None
        self.crop_end = None
        self.crop_rect = None
        self.canvas.config(cursor="arrow")
        
    def apply_mirror(self):
        """Applica l'effetto specchio (flip orizzontale)"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.original_image = self.current_image.transpose(Image.FLIP_LEFT_RIGHT)
        self.apply_adjustments()
        
    def apply_rotation(self):
        """Ruota l'immagine di 90 gradi in senso orario"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.original_image = self.current_image.rotate(-90, expand=True)
        self.apply_adjustments()
        
    def reset_image(self):
        """Resetta tutte le modifiche"""
        if self.original_image is None:
            return
            
        self.brightness_var.set(1.0)
        self.contrast_var.set(1.0)
        self.temperature_var.set(0)
        self.apply_adjustments()
        
    def save_image(self, format_type):
        """Salva l'immagine nel formato specificato"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Nessuna immagine da salvare!")
            return
            
        file_extension = f".{format_type}"
        file_types = [(format_type.upper(), f"*{file_extension}")]
        
        file_path = filedialog.asksaveasfilename(
            defaultextension=file_extension,
            filetypes=file_types,
            title=f"Salva come {format_type.upper()}"
        )
        
        if file_path:
            try:
                if format_type.lower() == 'jpg':
                    # Converti in RGB se necessario (JPG non supporta trasparenza)
                    img_to_save = self.current_image
                    if img_to_save.mode in ('RGBA', 'LA', 'P'):
                        rgb_img = Image.new('RGB', img_to_save.size, (255, 255, 255))
                        if img_to_save.mode == 'P':
                            img_to_save = img_to_save.convert('RGBA')
                        rgb_img.paste(img_to_save, mask=img_to_save.split()[-1] if img_to_save.mode == 'RGBA' else None)
                        img_to_save = rgb_img
                    img_to_save.save(file_path, 'JPEG', quality=95)
                else:
                    self.current_image.save(file_path, 'PNG')
                    
                messagebox.showinfo("Successo", f"Immagine salvata con successo in:\n{file_path}")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile salvare l'immagine:\n{str(e)}")


def main():
    """Funzione principale"""
    root = tk.Tk()
    app = PhotoEditor(root)
    
    # Configura il ridimensionamento della finestra
    root.minsize(800, 600)
    
    # Centra la finestra
    root.update_idletasks()
    width = root.winfo_width()
    height = root.winfo_height()
    x = (root.winfo_screenwidth() // 2) - (width // 2)
    y = (root.winfo_screenheight() // 2) - (height // 2)
    root.geometry(f'{width}x{height}+{x}+{y}')
    
    root.mainloop()


if __name__ == "__main__":
    main()


Prompt 2       
Va quasi tutto bene ma dobbiamo modificare i seguenti aspetti:       
- Per i pulsanti vintage e b/n il filtro deve poter essere applicato una volta sola, e se rischiaccio il bottone il filtro viene rimosso.     
- Fare in modo di indicare il bottone premuto o no, cambiando ad esempio lo sfondo del bottone.      
- Il bottone reset deve resettare TUTTI i filtri, non solo quelli dei 3 slider, e ritornare all'immagine originale.      
- Aggiungere il bottone Undo che annulla l'ultima modifica fatta.     

In [3]:
"""
Editor di Foto Professionale
Applicazione moderna per la modifica di immagini con interfaccia intuitiva
"""

import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from PIL import Image, ImageTk, ImageEnhance, ImageFilter, ImageOps
import numpy as np
from typing import Optional
import os


class PhotoEditor:
    """Editor di foto con interfaccia grafica moderna"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("Editor di Foto Professionale")
        self.root.geometry("1200x800")
        self.root.configure(bg="#1a1a1a")
        
        # Variabili per le immagini
        self.original_image: Optional[Image.Image] = None
        self.current_image: Optional[Image.Image] = None
        self.display_image: Optional[ImageTk.PhotoImage] = None
        self.crop_mode = False
        self.crop_start = None
        self.crop_rect = None
        
        # Cronologia per Undo (stack di immagini)
        self.history = []
        self.max_history = 20  # Massimo 20 stati nella cronologia
        
        # Variabili per i filtri
        self.brightness_var = tk.DoubleVar(value=1.0)
        self.contrast_var = tk.DoubleVar(value=1.0)
        self.temperature_var = tk.DoubleVar(value=0)
        
        # Stati dei filtri toggle
        self.vintage_active = False
        self.bw_active = False
        
        # Riferimenti ai pulsanti per cambio colore
        self.vintage_btn = None
        self.bw_btn = None
        
        # Configurazione dello stile moderno
        self.setup_styles()
        
        # Creazione dell'interfaccia
        self.create_widgets()
        
        # Binding degli eventi
        self.brightness_var.trace_add('write', self.apply_adjustments)
        self.contrast_var.trace_add('write', self.apply_adjustments)
        self.temperature_var.trace_add('write', self.apply_adjustments)
        
    def setup_styles(self):
        """Configura gli stili moderni per l'interfaccia"""
        style = ttk.Style()
        style.theme_use('clam')
        
        # Colori moderni
        bg_dark = "#1a1a1a"
        bg_medium = "#2d2d2d"
        bg_light = "#3d3d3d"
        accent = "#4a9eff"
        text_color = "#ffffff"
        
        # Stile per i pulsanti
        style.configure('Modern.TButton',
                       background=bg_medium,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10))
        style.map('Modern.TButton',
                 background=[('active', bg_light), ('pressed', accent)])
        
        # Stile per i pulsanti attivi (filtri applicati)
        style.configure('Active.TButton',
                       background=accent,
                       foreground=text_color,
                       borderwidth=0,
                       focuscolor='none',
                       padding=10,
                       font=('Segoe UI', 10, 'bold'))
        style.map('Active.TButton',
                 background=[('active', '#3a8eef'), ('pressed', '#2a7edf')])
        
        # Stile per le etichette
        style.configure('Modern.TLabel',
                       background=bg_dark,
                       foreground=text_color,
                       font=('Segoe UI', 10))
        
        # Stile per i frame
        style.configure('Modern.TFrame',
                       background=bg_dark)
        
        # Stile per gli slider
        style.configure('Modern.Horizontal.TScale',
                       background=bg_dark,
                       troughcolor=bg_medium,
                       borderwidth=0,
                       sliderthickness=20)
        
    def create_widgets(self):
        """Crea tutti i widget dell'interfaccia"""
        # Frame principale
        main_frame = ttk.Frame(self.root, style='Modern.TFrame')
        main_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Area superiore: canvas per l'immagine
        self.canvas_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        self.canvas_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 10))
        
        self.canvas = tk.Canvas(self.canvas_frame,
                               bg="#2d2d2d",
                               highlightthickness=0,
                               cursor="cross")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Testo placeholder
        self.canvas.create_text(
            400, 300,
            text="Carica un'immagine per iniziare",
            fill="#666666",
            font=('Segoe UI', 16),
            tags="placeholder"
        )
        
        # Binding per il crop
        self.canvas.bind("<Button-1>", self.on_crop_start)
        self.canvas.bind("<B1-Motion>", self.on_crop_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_crop_end)
        
        # Area inferiore: controlli
        controls_frame = ttk.Frame(main_frame, style='Modern.TFrame')
        controls_frame.pack(fill=tk.X)
        
        # Frame per i pulsanti principali
        buttons_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        buttons_frame.pack(fill=tk.X, pady=(0, 10))
        
        # Pulsante carica
        load_btn = ttk.Button(buttons_frame,
                             text="📁 Carica Immagine",
                             command=self.load_image,
                             style='Modern.TButton')
        load_btn.pack(side=tk.LEFT, padx=5)
        
        # Pulsante salva JPG
        save_jpg_btn = ttk.Button(buttons_frame,
                                 text="💾 Salva JPG",
                                 command=lambda: self.save_image('jpg'),
                                 style='Modern.TButton')
        save_jpg_btn.pack(side=tk.LEFT, padx=5)
        
        # Pulsante salva PNG
        save_png_btn = ttk.Button(buttons_frame,
                                 text="💾 Salva PNG",
                                 command=lambda: self.save_image('png'),
                                 style='Modern.TButton')
        save_png_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore
        ttk.Separator(buttons_frame, orient=tk.VERTICAL).pack(side=tk.LEFT, fill=tk.Y, padx=10)
        
        # Pulsanti filtri
        self.vintage_btn = ttk.Button(buttons_frame,
                                text="🎨 Vintage",
                                command=self.toggle_vintage,
                                style='Modern.TButton')
        self.vintage_btn.pack(side=tk.LEFT, padx=5)
        
        self.bw_btn = ttk.Button(buttons_frame,
                           text="⚫ Bianco e Nero",
                           command=self.toggle_bw,
                           style='Modern.TButton')
        self.bw_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore
        ttk.Separator(buttons_frame, orient=tk.VERTICAL).pack(side=tk.LEFT, fill=tk.Y, padx=10)
        
        # Pulsanti trasformazione
        crop_btn = ttk.Button(buttons_frame,
                             text="✂️ Ritaglia",
                             command=self.toggle_crop_mode,
                             style='Modern.TButton')
        crop_btn.pack(side=tk.LEFT, padx=5)
        
        mirror_btn = ttk.Button(buttons_frame,
                               text="🔄 Specchio",
                               command=self.apply_mirror,
                               style='Modern.TButton')
        mirror_btn.pack(side=tk.LEFT, padx=5)
        
        rotate_btn = ttk.Button(buttons_frame,
                               text="🔃 Ruota 90°",
                               command=self.apply_rotation,
                               style='Modern.TButton')
        rotate_btn.pack(side=tk.LEFT, padx=5)
        
        # Separatore
        ttk.Separator(buttons_frame, orient=tk.VERTICAL).pack(side=tk.LEFT, fill=tk.Y, padx=10)
        
        # Pulsante Undo
        undo_btn = ttk.Button(buttons_frame,
                             text="⮌ Undo",
                             command=self.undo,
                             style='Modern.TButton')
        undo_btn.pack(side=tk.LEFT, padx=5)
        
        # Pulsante reset
        reset_btn = ttk.Button(buttons_frame,
                              text="↺ Reset",
                              command=self.reset_image,
                              style='Modern.TButton')
        reset_btn.pack(side=tk.LEFT, padx=5)
        
        # Frame per gli slider
        sliders_frame = ttk.Frame(controls_frame, style='Modern.TFrame')
        sliders_frame.pack(fill=tk.X)
        
        # Slider luminosità
        self.create_slider(sliders_frame, "☀️ Luminosità", self.brightness_var, 0.0, 2.0, 0)
        
        # Slider contrasto
        self.create_slider(sliders_frame, "◐ Contrasto", self.contrast_var, 0.0, 2.0, 1)
        
        # Slider temperatura
        self.create_slider(sliders_frame, "🌡️ Temperatura", self.temperature_var, -100, 100, 2)
        
    def create_slider(self, parent, label_text, variable, from_, to, column):
        """Crea uno slider con etichetta"""
        frame = ttk.Frame(parent, style='Modern.TFrame')
        frame.grid(row=0, column=column, padx=10, pady=5, sticky='ew')
        parent.columnconfigure(column, weight=1)
        
        label = ttk.Label(frame, text=label_text, style='Modern.TLabel')
        label.pack(anchor='w')
        
        slider = ttk.Scale(frame,
                          from_=from_,
                          to=to,
                          variable=variable,
                          orient=tk.HORIZONTAL,
                          style='Modern.Horizontal.TScale')
        slider.pack(fill=tk.X, pady=5)
        
        # Etichetta valore
        value_label = ttk.Label(frame, text=f"{variable.get():.2f}", style='Modern.TLabel')
        value_label.pack(anchor='e')
        
        def update_label(*args):
            value_label.config(text=f"{variable.get():.2f}")
        
        variable.trace_add('write', update_label)
        
    def load_image(self):
        """Carica un'immagine dal file system"""
        file_path = filedialog.askopenfilename(
            title="Seleziona un'immagine",
            filetypes=[
                ("Immagini", "*.jpg *.jpeg *.png *.bmp *.gif"),
                ("JPEG", "*.jpg *.jpeg"),
                ("PNG", "*.png"),
                ("Tutti i file", "*.*")
            ]
        )
        
        if file_path:
            try:
                self.original_image = Image.open(file_path)
                self.current_image = self.original_image.copy()
                
                # Reset di tutti i controlli e filtri
                self.brightness_var.set(1.0)
                self.contrast_var.set(1.0)
                self.temperature_var.set(0)
                
                # Reset stato filtri
                self.vintage_active = False
                self.bw_active = False
                self.update_filter_buttons()
                
                # Reset cronologia
                self.history = []
                self.save_to_history()
                
                self.display_current_image()
                self.canvas.delete("placeholder")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile caricare l'immagine:\n{str(e)}")
                
    def display_current_image(self):
        """Visualizza l'immagine corrente sul canvas"""
        if self.current_image is None:
            return
            
        # Calcola le dimensioni per il fit
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        if canvas_width <= 1 or canvas_height <= 1:
            canvas_width = 800
            canvas_height = 600
            
        # Calcola il ridimensionamento mantenendo l'aspect ratio
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height)
        new_width = int(img_width * ratio * 0.9)  # 90% per margini
        new_height = int(img_height * ratio * 0.9)
        
        # Ridimensiona l'immagine
        display_img = self.current_image.resize((new_width, new_height), Image.Resampling.LANCZOS)
        
        # Converti in PhotoImage
        self.display_image = ImageTk.PhotoImage(display_img)
        
        # Pulisci il canvas e mostra l'immagine
        self.canvas.delete("all")
        self.canvas.create_image(
            canvas_width // 2,
            canvas_height // 2,
            image=self.display_image,
            anchor=tk.CENTER,
            tags="image"
        )
        
    def save_to_history(self):
        """Salva lo stato corrente nella cronologia"""
        if self.original_image is None:
            return
            
        # Salva una copia dell'immagine e dello stato dei filtri
        state = {
            'image': self.original_image.copy(),
            'brightness': self.brightness_var.get(),
            'contrast': self.contrast_var.get(),
            'temperature': self.temperature_var.get(),
            'vintage': self.vintage_active,
            'bw': self.bw_active
        }
        
        self.history.append(state)
        
        # Limita la dimensione della cronologia
        if len(self.history) > self.max_history:
            self.history.pop(0)
            
    def undo(self):
        """Annulla l'ultima modifica"""
        if len(self.history) <= 1:
            messagebox.showinfo("Undo", "Nessuna operazione da annullare!")
            return
            
        # Rimuovi lo stato corrente
        self.history.pop()
        
        # Ripristina lo stato precedente
        if self.history:
            previous_state = self.history[-1]
            self.original_image = previous_state['image'].copy()
            self.brightness_var.set(previous_state['brightness'])
            self.contrast_var.set(previous_state['contrast'])
            self.temperature_var.set(previous_state['temperature'])
            self.vintage_active = previous_state['vintage']
            self.bw_active = previous_state['bw']
            
            self.update_filter_buttons()
            self.apply_adjustments()
            
    def update_filter_buttons(self):
        """Aggiorna lo stile dei pulsanti filtro in base allo stato"""
        if self.vintage_btn:
            style = 'Active.TButton' if self.vintage_active else 'Modern.TButton'
            self.vintage_btn.configure(style=style)
            
        if self.bw_btn:
            style = 'Active.TButton' if self.bw_active else 'Modern.TButton'
            self.bw_btn.configure(style=style)
        
    def apply_adjustments(self, *args):
        """Applica le regolazioni di luminosità, contrasto e temperatura"""
        if self.original_image is None:
            return
            
        # Parte dall'immagine originale
        img = self.original_image.copy()
        
        # Applica luminosità
        if self.brightness_var.get() != 1.0:
            enhancer = ImageEnhance.Brightness(img)
            img = enhancer.enhance(self.brightness_var.get())
            
        # Applica contrasto
        if self.contrast_var.get() != 1.0:
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(self.contrast_var.get())
            
        # Applica temperatura (modifica del bilanciamento del colore)
        if self.temperature_var.get() != 0:
            img = self.adjust_temperature(img, self.temperature_var.get())
            
        self.current_image = img
        self.display_current_image()
        
    def adjust_temperature(self, image, value):
        """Regola la temperatura del colore dell'immagine"""
        if image.mode != 'RGB':
            image = image.convert('RGB')
            
        # Converti in array numpy
        img_array = np.array(image, dtype=np.float32)
        
        # Applica shift di temperatura
        if value > 0:  # Più caldo (più rosso/giallo)
            img_array[:, :, 0] += value * 0.5  # Rosso
            img_array[:, :, 1] += value * 0.3  # Verde
        else:  # Più freddo (più blu)
            img_array[:, :, 2] += abs(value) * 0.5  # Blu
            
        # Clamp values
        img_array = np.clip(img_array, 0, 255)
        
        return Image.fromarray(img_array.astype(np.uint8))
        
    def toggle_vintage(self):
        """Attiva/disattiva il filtro vintage (seppia)"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Salva lo stato corrente prima di modificare
        self.save_to_history()
        
        # Toggle dello stato
        self.vintage_active = not self.vintage_active
        
        if self.vintage_active:
            # Disattiva B/N se attivo
            if self.bw_active:
                self.bw_active = False
            
            # Applica filtro vintage
            img = self.original_image.convert('RGB')
            img_array = np.array(img, dtype=np.float32)
            
            # Matrice seppia
            sepia_filter = np.array([
                [0.393, 0.769, 0.189],
                [0.349, 0.686, 0.168],
                [0.272, 0.534, 0.131]
            ])
            
            # Applica il filtro
            sepia_img = img_array @ sepia_filter.T
            sepia_img = np.clip(sepia_img, 0, 255)
            
            self.original_image = Image.fromarray(sepia_img.astype(np.uint8))
        else:
            # Rimuovi il filtro tornando all'immagine dalla cronologia
            if len(self.history) >= 2:
                # Trova l'ultimo stato senza vintage
                for i in range(len(self.history) - 2, -1, -1):
                    if not self.history[i]['vintage']:
                        self.original_image = self.history[i]['image'].copy()
                        break
        
        self.update_filter_buttons()
        self.apply_adjustments()
        
    def toggle_bw(self):
        """Attiva/disattiva il filtro bianco e nero"""
        if self.original_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        # Salva lo stato corrente prima di modificare
        self.save_to_history()
        
        # Toggle dello stato
        self.bw_active = not self.bw_active
        
        if self.bw_active:
            # Disattiva Vintage se attivo
            if self.vintage_active:
                self.vintage_active = False
            
            # Applica filtro bianco e nero
            self.original_image = self.original_image.convert('L').convert('RGB')
        else:
            # Rimuovi il filtro tornando all'immagine dalla cronologia
            if len(self.history) >= 2:
                # Trova l'ultimo stato senza b/n
                for i in range(len(self.history) - 2, -1, -1):
                    if not self.history[i]['bw']:
                        self.original_image = self.history[i]['image'].copy()
                        break
        
        self.update_filter_buttons()
        self.apply_adjustments()
        
    def toggle_crop_mode(self):
        """Attiva/disattiva la modalità ritaglio"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
            
        self.crop_mode = not self.crop_mode
        if self.crop_mode:
            self.canvas.config(cursor="cross")
            messagebox.showinfo("Modalità Ritaglio",
                              "Trascina il mouse sull'immagine per selezionare l'area da ritagliare.\n"
                              "Clicca di nuovo su 'Ritaglia' per applicare.")
        else:
            self.canvas.config(cursor="arrow")
            if self.crop_rect:
                self.apply_crop()
                
    def on_crop_start(self, event):
        """Inizia la selezione per il ritaglio"""
        if not self.crop_mode:
            return
        self.crop_start = (event.x, event.y)
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
    def on_crop_drag(self, event):
        """Disegna il rettangolo di selezione"""
        if not self.crop_mode or not self.crop_start:
            return
            
        if self.crop_rect:
            self.canvas.delete(self.crop_rect)
            
        self.crop_rect = self.canvas.create_rectangle(
            self.crop_start[0], self.crop_start[1],
            event.x, event.y,
            outline='#4a9eff',
            width=2,
            dash=(5, 5)
        )
        
    def on_crop_end(self, event):
        """Finalizza la selezione"""
        if not self.crop_mode or not self.crop_start:
            return
        self.crop_end = (event.x, event.y)
        
    def apply_crop(self):
        """Applica il ritaglio all'immagine"""
        if not self.crop_start or not self.crop_end:
            return
            
        # Calcola le coordinate relative all'immagine
        canvas_width = self.canvas.winfo_width()
        canvas_height = self.canvas.winfo_height()
        
        img_width, img_height = self.current_image.size
        ratio = min(canvas_width / img_width, canvas_height / img_height) * 0.9
        
        display_width = int(img_width * ratio)
        display_height = int(img_height * ratio)
        
        offset_x = (canvas_width - display_width) // 2
        offset_y = (canvas_height - display_height) // 2
        
        # Converti coordinate canvas in coordinate immagine
        x1 = int((min(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y1 = int((min(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        x2 = int((max(self.crop_start[0], self.crop_end[0]) - offset_x) / ratio)
        y2 = int((max(self.crop_start[1], self.crop_end[1]) - offset_y) / ratio)
        
        # Clamp ai limiti dell'immagine
        x1 = max(0, min(x1, img_width))
        y1 = max(0, min(y1, img_height))
        x2 = max(0, min(x2, img_width))
        y2 = max(0, min(y2, img_height))
        
        if x2 > x1 and y2 > y1:
            self.save_to_history()
            self.original_image = self.current_image.crop((x1, y1, x2, y2))
            self.apply_adjustments()
            
        self.crop_mode = False
        self.crop_start = None
        self.crop_end = None
        self.crop_rect = None
        self.canvas.config(cursor="arrow")
        
    def apply_mirror(self):
        """Applica l'effetto specchio (flip orizzontale)"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.save_to_history()
        self.original_image = self.current_image.transpose(Image.FLIP_LEFT_RIGHT)
        self.apply_adjustments()
        
    def apply_rotation(self):
        """Ruota l'immagine di 90 gradi in senso orario"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Carica prima un'immagine!")
            return
        
        self.save_to_history()
        self.original_image = self.current_image.rotate(-90, expand=True)
        self.apply_adjustments()
        
    def reset_image(self):
        """Resetta tutte le modifiche e torna all'immagine originale"""
        if not self.history:
            return
        
        # Torna alla primissima immagine caricata
        first_state = self.history[0]
        self.original_image = first_state['image'].copy()
        
        # Reset di tutti i controlli
        self.brightness_var.set(1.0)
        self.contrast_var.set(1.0)
        self.temperature_var.set(0)
        
        # Reset di tutti i filtri
        self.vintage_active = False
        self.bw_active = False
        self.update_filter_buttons()
        
        # Resetta la cronologia mantenendo solo lo stato originale
        self.history = [first_state]
        
        self.apply_adjustments()
        
    def save_image(self, format_type):
        """Salva l'immagine nel formato specificato"""
        if self.current_image is None:
            messagebox.showwarning("Attenzione", "Nessuna immagine da salvare!")
            return
            
        file_extension = f".{format_type}"
        file_types = [(format_type.upper(), f"*{file_extension}")]
        
        file_path = filedialog.asksaveasfilename(
            defaultextension=file_extension,
            filetypes=file_types,
            title=f"Salva come {format_type.upper()}"
        )
        
        if file_path:
            try:
                if format_type.lower() == 'jpg':
                    # Converti in RGB se necessario (JPG non supporta trasparenza)
                    img_to_save = self.current_image
                    if img_to_save.mode in ('RGBA', 'LA', 'P'):
                        rgb_img = Image.new('RGB', img_to_save.size, (255, 255, 255))
                        if img_to_save.mode == 'P':
                            img_to_save = img_to_save.convert('RGBA')
                        rgb_img.paste(img_to_save, mask=img_to_save.split()[-1] if img_to_save.mode == 'RGBA' else None)
                        img_to_save = rgb_img
                    img_to_save.save(file_path, 'JPEG', quality=95)
                else:
                    self.current_image.save(file_path, 'PNG')
                    
                messagebox.showinfo("Successo", f"Immagine salvata con successo in:\n{file_path}")
                
            except Exception as e:
                messagebox.showerror("Errore", f"Impossibile salvare l'immagine:\n{str(e)}")


def main():
    """Funzione principale"""
    root = tk.Tk()
    app = PhotoEditor(root)
    
    # Configura il ridimensionamento della finestra
    root.minsize(800, 600)
    
    # Centra la finestra
    root.update_idletasks()
    width = root.winfo_width()
    height = root.winfo_height()
    x = (root.winfo_screenwidth() // 2) - (width // 2)
    y = (root.winfo_screenheight() // 2) - (height // 2)
    root.geometry(f'{width}x{height}+{x}+{y}')
    
    root.mainloop()


if __name__ == "__main__":
    main()
